# Preprocess NIH ChestX-ray14

Reads NIH ChestX-ray14, keeps the official test list, and holds out about 10% of the remaining patients for validation. Every selected image is resized once to 224x224 and stored as uint8 grayscale in sharded TFRecords under OUTPUT_DIR (default data/output), so that output can be reused by the training notebooks.

Label columns use one fixed sorted order (14 findings plus No Finding). That order is written to `label_order.json` and checked again at train and eval time.


In [ ]:
import os

SEED = 42
DATA_DIR = os.environ.get("DATA_DIR", "data/nih")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "data/output")

if os.environ.get("CXR_SEED"):
    SEED = int(os.environ["CXR_SEED"])
print("SEED", SEED)
print("DATA_DIR", DATA_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)


In [ ]:
import sys
from pathlib import Path

def add_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *list(here.parents)[:4]]
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "src" / "__init__.py").is_file() and (resolved / "src" / "paths.py").is_file():
            root = str(resolved)
            if root not in sys.path:
                sys.path.insert(0, root)
            print("Using code at", resolved)
            return resolved
    raise FileNotFoundError(
        "Could not find src/paths.py. Run this notebook from the project directory."
    )

add_project_root()


In [ ]:
from src.preprocess import run_preprocess

summary = run_preprocess(seed=SEED, data_dir=DATA_DIR, out_dir=OUTPUT_DIR)
print("Images written:", summary["n_images"])
print("Output directory:", summary["out_dir"])
